In [1]:
import numpy as np
import cv2

In [17]:
import numpy as np
import cv2


webcam = cv2.VideoCapture(0)
window_name = "window"
cv2.namedWindow(window_name,cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, (800,600))

while True: 
    success, imageFrame = webcam.read() 
    if not success:
        break

    # Convert to HSV color space
    hsvFrame = cv2.cvtColor(imageFrame, cv2.COLOR_BGR2HSV) 

    # --- Define Color Ranges ---
    # Red mask
    red_lower = np.array([136, 87, 111], np.uint8) 
    red_upper = np.array([180, 255, 255], np.uint8) 
    red_mask = cv2.inRange(hsvFrame, red_lower, red_upper) 

    # Green mask
    green_lower = np.array([25, 52, 72], np.uint8) 
    green_upper = np.array([102, 255, 255], np.uint8) 
    green_mask = cv2.inRange(hsvFrame, green_lower, green_upper) 

    # Blue mask
    blue_lower = np.array([94, 80, 2], np.uint8) 
    blue_upper = np.array([120, 255, 255], np.uint8) 
    blue_mask = cv2.inRange(hsvFrame, blue_lower, blue_upper) 

    # Cleaning masks using Dilation
    kernel = np.ones((5, 5), "uint8") 
    red_mask = cv2.dilate(red_mask, kernel) 
    green_mask = cv2.dilate(green_mask, kernel) 
    blue_mask = cv2.dilate(blue_mask, kernel)

    def track_color(mask, label, color_bgr):
        contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            if cv2.contourArea(contour) > 500:
                x, y, w, h = cv2.boundingRect(contour)
                # Draw the rectangle
                cv2.rectangle(imageFrame, (x, y), (x + w, y + h), color_bgr, 2)
                # Draw the label above the rectangle
                cv2.putText(imageFrame, label, (x, y - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, color_bgr, 2)

    # Execute tracking
    track_color(red_mask, "Red", (0, 0, 255))
    track_color(green_mask, "Green", (0, 255, 0))
    track_color(blue_mask, "Blue", (255, 0, 0))
    
    # Show the output in the named window
    cv2.imshow(window_name, imageFrame) 
    
    # Exit on 'q' key
    if cv2.waitKey(1) & 0xFF == ord('q'): 
        break

webcam.release() 
cv2.destroyAllWindows()

[ WARN:0@2045.698] global cap_gstreamer.cpp:1173 isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created
